In [4]:
import polars as pl
import pandas as pd

# File Paths
vcf_path = "toy_trimmed2.vcf" 
results_path = "matched_snps_output2.txt" 

# === Load VCF ===
vcf = pl.read_csv(vcf_path, separator="\t", has_header=True, comment_prefix="##")
vcf = vcf.rename({"#CHROM": "CHROM"})

# ALT genotypes of interest
alt_genos = {"0/1", "1/0", "1/1"}

# Load predictions
predictions = pd.read_csv(results_path, sep="\t")
tissues = ["RNA-seq: Putamen", "RNA-seq: Caudate"]

# Initialize: {tissue: list of rows}
tissue_long_rows = {t: [] for t in tissues}

print(f"VCF shape: {vcf.shape}")
print(f"Predictions shape: {predictions.shape}")

VCF shape: (10, 13)
Predictions shape: (20, 5)


Printing all the predicions

In [5]:
print(predictions)

                snp             gene alt_allele            tissue    logSED
0    18_9806921_A_G  ENSG00000168461          G  RNA-seq: Putamen  0.000712
1    18_9806921_A_G  ENSG00000168461          G  RNA-seq: Caudate  0.000730
2    18_9806921_A_G  ENSG00000285653          G  RNA-seq: Putamen  0.000267
3    18_9806921_A_G  ENSG00000285653          G  RNA-seq: Caudate  0.000026
4    18_9806921_A_G  ENSG00000168454          G  RNA-seq: Putamen  0.000246
5    18_9806921_A_G  ENSG00000168454          G  RNA-seq: Caudate  0.000822
6   18_11979422_G_A  ENSG00000141404          A  RNA-seq: Putamen  0.000575
7   18_11979422_G_A  ENSG00000141404          A  RNA-seq: Caudate  0.000441
8   18_11979422_G_A  ENSG00000154889          A  RNA-seq: Putamen  0.000186
9   18_11979422_G_A  ENSG00000154889          A  RNA-seq: Caudate -0.000032
10  18_11979422_G_A  ENSG00000273141          A  RNA-seq: Putamen  0.000373
11  18_11979422_G_A  ENSG00000273141          A  RNA-seq: Caudate -0.000318
12  18_11979

Printing how the VCF File looks 

In [6]:
print(vcf)

shape: (10, 13)
┌───────┬──────────┬─────────────┬─────┬───┬─────────────┬─────────────┬─────────────┬─────────────┐
│ CHROM ┆ POS      ┆ ID          ┆ REF ┆ … ┆ gwa1.100_gw ┆ gwa1.1012_g ┆ gwa1.1016_g ┆ gwa1.101_gw │
│ ---   ┆ ---      ┆ ---         ┆ --- ┆   ┆ a1.100      ┆ wa1.1012    ┆ wa1.1016    ┆ a1.101      │
│ str   ┆ i64      ┆ str         ┆ str ┆   ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│       ┆          ┆             ┆     ┆   ┆ str         ┆ str         ┆ str         ┆ str         │
╞═══════╪══════════╪═════════════╪═════╪═══╪═════════════╪═════════════╪═════════════╪═════════════╡
│ chr18 ┆ 9806924  ┆ 18_9806921_ ┆ A   ┆ … ┆ 0/0         ┆ 0/1         ┆ 0/0         ┆ 0/1         │
│       ┆          ┆ A_G         ┆     ┆   ┆             ┆             ┆             ┆             │
│ chr18 ┆ 11979423 ┆ 18_11979422 ┆ G   ┆ … ┆ 0/0         ┆ 0/1         ┆ 0/0         ┆ 0/1         │
│       ┆          ┆ _G_A        ┆     ┆   ┆             ┆             ┆   

I tested the same analysis with less data such as 10 rows in the VCF file and 50 rows in the predictions but the problem was that there ware not more than one SNP affecting the same gene so there was not a lot to see

In [7]:
meta_cols = {"CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO", "FORMAT"}
sample_names = [col for col in vcf.columns if col not in meta_cols]
print(f"Samples found: {sample_names}")


Samples found: ['gwa1.100_gwa1.100', 'gwa1.1012_gwa1.1012', 'gwa1.1016_gwa1.1016', 'gwa1.101_gwa1.101']


Now printing  how many SNPS have an ALT alelle for each Sample and a list of these SNPS

In [4]:
sample_snps = {}

for sample_name in sample_names:
    snps = vcf.filter(pl.col(sample_name).is_in(alt_genos)).select("ID").to_series().to_list()
    snps = set(str(s).strip() for s in snps)
    sample_snps[sample_name] = snps
    print(f"\nSample: {sample_name}")
    print(f"Number of ALT SNPs: {len(snps)}")
    print(f"ALT SNPs: {sorted(snps)}")




Sample: gwa1.100_gwa1.100
Number of ALT SNPs: 4
ALT SNPs: ['18_10421654_A_G', '18_32956524_G_A', '18_55110898_G_C', '18_74103092_T_C']

Sample: gwa1.1012_gwa1.1012
Number of ALT SNPs: 5
ALT SNPs: ['18_10421654_A_G', '18_11979422_G_A', '18_47494393_G_A', '18_74103092_T_C', '18_9806921_A_G']

Sample: gwa1.1016_gwa1.1016
Number of ALT SNPs: 3
ALT SNPs: ['18_10421654_A_G', '18_47494393_G_A', '18_74103092_T_C']

Sample: gwa1.101_gwa1.101
Number of ALT SNPs: 4
ALT SNPs: ['18_11979422_G_A', '18_32956524_G_A', '18_47494393_G_A', '18_9806921_A_G']


Now just for the first sample i am printing all the matched predictions for both tissues 
Test can be done for the other 3 samples by changing the sample here:
sample_to_check = sample_names[0] 

In [9]:
sample_to_check = sample_names[1] 
snps = sample_snps[sample_to_check]

sample_preds = predictions[predictions["snp"].isin(snps)]
print(f"\nSample: {sample_to_check}")
print(f"Matching prediction rows: {len(sample_preds)}")
print(sample_preds)



Sample: gwa1.1012_gwa1.1012
Matching prediction rows: 20
                snp             gene alt_allele            tissue    logSED
0    18_9806921_A_G  ENSG00000168461          G  RNA-seq: Putamen  0.000712
1    18_9806921_A_G  ENSG00000168461          G  RNA-seq: Caudate  0.000730
2    18_9806921_A_G  ENSG00000285653          G  RNA-seq: Putamen  0.000267
3    18_9806921_A_G  ENSG00000285653          G  RNA-seq: Caudate  0.000026
4    18_9806921_A_G  ENSG00000168454          G  RNA-seq: Putamen  0.000246
5    18_9806921_A_G  ENSG00000168454          G  RNA-seq: Caudate  0.000822
6   18_11979422_G_A  ENSG00000141404          A  RNA-seq: Putamen  0.000575
7   18_11979422_G_A  ENSG00000141404          A  RNA-seq: Caudate  0.000441
8   18_11979422_G_A  ENSG00000154889          A  RNA-seq: Putamen  0.000186
9   18_11979422_G_A  ENSG00000154889          A  RNA-seq: Caudate -0.000032
10  18_11979422_G_A  ENSG00000273141          A  RNA-seq: Putamen  0.000373
11  18_11979422_G_A  ENSG00000

Printing the final logSED sum for the tissue Putamen for the same sample only

In [21]:
#for tissue in tissues:
tissue_preds = sample_preds[sample_preds["tissue"] == tissues[0]]
print(f"\nTissue: {tissues[0]}")
print(f"Tissue predictions: {len(tissue_preds)}")
    
   

grouped = tissue_preds.groupby("gene")["logSED"].sum().reset_index()
print(f"Aggregated expression vector for {sample_to_check} - {tissues[0]}")
print(grouped)



Tissue: RNA-seq: Putamen
Tissue predictions: 10
Aggregated expression vector for gwa1.1012_gwa1.1012 - RNA-seq: Putamen
              gene    logSED
0  ENSG00000141401 -0.000585
1  ENSG00000141404  0.000575
2  ENSG00000154889  0.000186
3  ENSG00000168454  0.000246
4  ENSG00000168461  0.000712
5  ENSG00000266955  0.000177
6  ENSG00000267079  0.000272
7  ENSG00000267480  0.000277
8  ENSG00000273141  0.000373
9  ENSG00000285653  0.000267


For the same Sample AND tissue again :
Printing for each Gene 
A list of the snps that contributed to the sum plus the logSED values for validation of the sum


In [22]:
# Print detailed contribution for each gene
for gene, group in tissue_preds.groupby("gene"):
    snp_ids = group["snp"].tolist()
    logsed_vals = group["logSED"].tolist()


    total = sum(logsed_vals)

    print(f"\nGene: {gene}")
    print(f"  SNPs used:     {', '.join(snp_ids)}")
    print(f"  logSED values: {', '.join([str(round(v, 7)) for v in logsed_vals])}")
    print(f"  Sum:           {round(total, 7)}")



Gene: ENSG00000141401
  SNPs used:     18_11979422_G_A
  logSED values: -0.0005846
  Sum:           -0.0005846

Gene: ENSG00000141404
  SNPs used:     18_11979422_G_A
  logSED values: 0.0005746
  Sum:           0.0005746

Gene: ENSG00000154889
  SNPs used:     18_11979422_G_A
  logSED values: 0.0001857
  Sum:           0.0001857

Gene: ENSG00000168454
  SNPs used:     18_9806921_A_G
  logSED values: 0.000246
  Sum:           0.000246

Gene: ENSG00000168461
  SNPs used:     18_9806921_A_G
  logSED values: 0.0007124
  Sum:           0.0007124

Gene: ENSG00000266955
  SNPs used:     18_11979422_G_A
  logSED values: 0.0001768
  Sum:           0.0001768

Gene: ENSG00000267079
  SNPs used:     18_11979422_G_A
  logSED values: 0.0002718
  Sum:           0.0002718

Gene: ENSG00000267480
  SNPs used:     18_11979422_G_A
  logSED values: 0.0002768
  Sum:           0.0002768

Gene: ENSG00000273141
  SNPs used:     18_11979422_G_A
  logSED values: 0.0003731
  Sum:           0.0003731

Gene: ENSG0

Printing the final logSED sum for the tissue Caudate for the same sample only

In [13]:
tissue_preds = sample_preds[sample_preds["tissue"] == tissues[1]]
print(f"\nTissue: {tissues[1]}")
print(f"Tissue predictions: {len(tissue_preds)}")
    
   

grouped = tissue_preds.groupby("gene")["logSED"].sum().reset_index()
print(f"Aggregated expression vector for {sample_to_check} - {tissues[1]}")
print(grouped)



Tissue: RNA-seq: Caudate
Tissue predictions: 10
Aggregated expression vector for gwa1.1012_gwa1.1012 - RNA-seq: Caudate
              gene    logSED
0  ENSG00000141401  0.000919
1  ENSG00000141404  0.000441
2  ENSG00000154889 -0.000032
3  ENSG00000168454  0.000822
4  ENSG00000168461  0.000730
5  ENSG00000266955 -0.000409
6  ENSG00000267079  0.000280
7  ENSG00000267480  0.000143
8  ENSG00000273141 -0.000318
9  ENSG00000285653  0.000026


For the same Sample AND tissue again:
Printing for each Gene 
A list of the snps that contributed to the sum plus the logSED values for validation of the sum
Prints only if there is 2 or more snps afecting

In [16]:
# Print detailed contribution for each gene
for gene, group in tissue_preds.groupby("gene"):
    snp_ids = group["snp"].tolist()
    logsed_vals = group["logSED"].tolist()

    total = sum(logsed_vals)

    print(f"\nGene: {gene}")
    print(f"  SNPs used:     {', '.join(snp_ids)}")
    print(f"  logSED values: {', '.join([str(round(v, 7)) for v in logsed_vals])}")
    print(f"  Sum:           {round(total, 7)}")


Gene: ENSG00000141401
  SNPs used:     18_11979422_G_A
  logSED values: 0.0009193
  Sum:           0.0009193

Gene: ENSG00000141404
  SNPs used:     18_11979422_G_A
  logSED values: 0.000441
  Sum:           0.000441

Gene: ENSG00000154889
  SNPs used:     18_11979422_G_A
  logSED values: -3.25e-05
  Sum:           -3.25e-05

Gene: ENSG00000168454
  SNPs used:     18_9806921_A_G
  logSED values: 0.000822
  Sum:           0.000822

Gene: ENSG00000168461
  SNPs used:     18_9806921_A_G
  logSED values: 0.00073
  Sum:           0.00073

Gene: ENSG00000266955
  SNPs used:     18_11979422_G_A
  logSED values: -0.000409
  Sum:           -0.000409

Gene: ENSG00000267079
  SNPs used:     18_11979422_G_A
  logSED values: 0.0002804
  Sum:           0.0002804

Gene: ENSG00000267480
  SNPs used:     18_11979422_G_A
  logSED values: 0.0001433
  Sum:           0.0001433

Gene: ENSG00000273141
  SNPs used:     18_11979422_G_A
  logSED values: -0.000318
  Sum:           -0.000318

Gene: ENSG000002856